# Chapter 1 — Empirical Characterization of Oil–Sovereign Transmission

This notebook merges the four working notebooks of Chapter 1 into a single, reviewer-friendly
document. It is organized in four sections:

1. **Experimental groups** — definition and characterization of oil exporters vs. controls,
   including placebo classifications.
2. **Cross-commodity correlations** — Brent vs. other commodities (cash and 12-month futures).
3. **Empirical characterization** — OVX-conditioned panel regressions and futures-basis regressions.
4. **Jump characterization & estimation** — GARCH(1,1) vs. GARCH-Jump LR tests on oil, CDS and
   MSCI series, plus daily/weekly jump detection and OVX → λ calibration.

---


# 1. Experimental Groups


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import datetime

# Constants
BARRELS_PER_TONNE = 7.33
TONNES_PER_MT = 1_000_000

In [ ]:
MSCI_EM_constituent_list = [ "Brazil", "Chile", "China", "Colombia", "Czechia", "Egypt",
                             "Greece", "Hungary", "India", "Indonesia", "South Korea", "Kuwait", 
                             "Malaysia", "Mexico", "Peru", "Philippines", "Poland", "Qatar",
                            "Saudi Arabia", "South Africa", "Taiwan", "Thailand", "Turkey", "United Arab Emirates"]

MSCI_EM_constituent_list_clean = [ "Brazil", "Chile", "China", "Colombia", "Czechia", "Egypt",
                             "Greece", "Hungary", "India", "Indonesia", "South Korea", "Kuwait", 
                             "Malaysia", "Mexico", "Peru", "Philippines", "Poland", "Qatar",
                            "Saudi Arabia", "South Africa", "Taiwan", "Thailand", "Turkey", "Abu Dhabi", "Dubai"] #Length: 24

### Valid CDS data

In [ ]:
CDS_df = pd.read_csv('../data/processed/CDS/Weekly_CDS.csv')
CDS_df['Date'] = pd.to_datetime(CDS_df['Date'])
CDS_df.set_index('Date', inplace=True)

CDS_df = CDS_df[CDS_df.index >= datetime.datetime(2014,1,1)].copy()
CDS_df = CDS_df[[col for col in CDS_df.columns if col in MSCI_EM_constituent_list_clean]]

cds_stats = []

for country in CDS_df.columns:
    # Get the time series for this country (drop NaN values)
    series = CDS_df[country].dropna()
    
    if len(series) > 0:
        # Start date
        start_date = series.index[0]
        
        # Calculate returns (percentage change from one row to next)
        returns = series.diff()
        
        # Count zero returns (where consecutive values are the same)
        zero_returns = (returns == 0).sum()
        
        # Percentage of zero returns
        total_returns = len(returns) - 1  # Exclude the first NaN from diff()
        pct_zero_returns = (zero_returns / total_returns * 100) if total_returns > 0 else 0
        
        cds_stats.append({
            'Country': country,
            'Series Start': start_date,
            'Observations': len(series),
            'Zero Returns (%)': round(pct_zero_returns, 2),
            'Decision': "Keep" if pct_zero_returns < 5 else "Discard"
        })

# Create and display the statistics table
cds_stats_df = pd.DataFrame(cds_stats).sort_values('Country')
print(cds_stats_df.to_string(index=False))
print(f"\nTotal countries: {len(cds_stats_df)}")



In [ ]:
# Crude oil balance (need to handle apostrophe thousands separator)
crude_oil_balance = pd.read_csv('../data/processed/Oil/crude_oil_trade_balance.csv', 
                                 thousands="'")
crude_oil_balance.set_index('Country', inplace=True)

crude_filtered = crude_oil_balance.loc[crude_oil_balance.index.isin(MSCI_EM_constituent_list)]

crude_filtered = crude_filtered.T
crude_filtered.index = crude_filtered.index.astype(int)

# Oil prices
annual_oil_price = pd.read_csv('../data/processed/Oil/Annual_avg_price_of_oil_macrotrends.csv')
annual_oil_price['Year'] = pd.to_datetime(annual_oil_price['Date']).dt.year
oil_prices = annual_oil_price.set_index('Year')['Value']

In [ ]:
# -----------------------------------------------------------------------------
# CONVERT MT TO USD BILLIONS
# -----------------------------------------------------------------------------

# After loading and filtering crude oil balance, convert to numeric
crude_filtered = crude_filtered.apply(pd.to_numeric, errors='coerce')

crude_oil_usd = crude_filtered.copy()
# Then do the conversion
for year in crude_oil_usd.index:
    if year in oil_prices.index:
        price = oil_prices[year]
        crude_oil_usd.loc[year] = crude_filtered.loc[year] * TONNES_PER_MT * BARRELS_PER_TONNE * price / 1e9

In [ ]:
# -----------------------------------------------------------------------------
# PREPARE GDP (map country names and convert to billions)
# -----------------------------------------------------------------------------

# GDP (long format)
IMFWEO_data = pd.read_csv('../data/processed/Macroeconomic_variables/IMF_WEO_full_annual_data.csv', delimiter=';')

#Filter selected columns
gdp_filtered = IMFWEO_data[IMFWEO_data['INDICATOR'] == 'Gross domestic product (GDP) - Current prices - US dollar'].drop(columns=['INDICATOR','FREQUENCY','SCALE'])

#Set index
gdp_filtered.set_index('COUNTRY',inplace=True)

# Filter to selected countries
gdp_filtered = gdp_filtered.loc[gdp_filtered.index.isin(MSCI_EM_constituent_list)]

# Transpose: index = years, columns = countries
gdp_transposed = gdp_filtered.T
gdp_transposed.index = gdp_transposed.index.astype(int)

#gdp_transposed
gdp_transposed.head()

In [ ]:
# -----------------------------------------------------------------------------
# CALCULATE OIL BALANCE AS % OF GDP
# -----------------------------------------------------------------------------

common_years = crude_oil_usd.index.intersection(gdp_transposed.index)
common_countries = crude_oil_usd.columns.intersection(gdp_transposed.columns)

print(f"\nCommon years: {common_years.min()} - {common_years.max()}")
print(f"Common countries: {list(common_countries)}")

crude_aligned = crude_oil_usd.loc[common_years, common_countries]
gdp_aligned = gdp_transposed.loc[common_years, common_countries]

oil_balance_pct_gdp = (crude_aligned / gdp_aligned) * 100


# -----------------------------------------------------------------------------
# SUMMARY TABLE
# -----------------------------------------------------------------------------
start_year = 2014
end_year = 2024

recent_years = [y for y in range(start_year, end_year + 1) if y in oil_balance_pct_gdp.index]
avg_balance = oil_balance_pct_gdp.loc[recent_years].mean().sort_values(ascending=True)  # ascending so exporters (negative) at top

print("\n" + "="*60)
print(f"AVERAGE CRUDE OIL TRADE BALANCE (% of GDP), {start_year}-{end_year}")
print("="*60)

for country in avg_balance.index:
    print(f"{country:20s}: {avg_balance[country]:+.2f}%")


In [ ]:
# -----------------------------------------------------------------------------
# PLOT
# -----------------------------------------------------------------------------

oil_exporters = ['Kuwait','Saudi Arabia', 'United Arab Emirates','Qatar','Colombia','Mexico','Brazil','Egypt','Malaysia']
controls = ['Indonesia','Philippines','Czechia','Turkey','Chile','China','South Africa',
            'Poland','Taiwan','India','South Korea','Tailand']

oil_balance_plot = oil_balance_pct_gdp.loc[start_year:end_year]

fig, ax = plt.subplots(figsize=(14, 8))

for country in controls:
    if country in oil_balance_plot.columns:
        ax.plot(oil_balance_plot.index, oil_balance_plot[country], 
                's--', color='blue', linewidth=1.5, markersize=4, label=country)
        
for country in oil_exporters:
    if country in oil_balance_plot.columns:
        ax.plot(oil_balance_plot.index, oil_balance_plot[country], 
                's--', color='red', linewidth=1.5, markersize=4, label=country)

ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Crude Oil Balance (% of GDP)', fontsize=12)
ax.set_title(f'Crude Oil Trade Balance as % of GDP ({start_year}-{end_year})\nRed = Oil Exporters, Blue = Controls', fontsize=14)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Proposed groups

In [ ]:
oil_exporters = ['Saudi Arabia', 'United Arab Emirates','Qatar','Colombia','Mexico','Brazil','Egypt','Malaysia']
controls = ['Indonesia','Philippines','Turkey','Chile','China','South Africa','South Korea','Tailand'] 

benchmarks = ['Emerging Market and Developing Economies', 'World']


### Systematic Differences between groups that could lead to confounding?

- Geopolitical / Geography
- Credit Ratings
- GDP per Capita
- Debt to GDP
- Current Account
- 


In [ ]:
all_countries = oil_exporters + controls + benchmarks

variables_of_interest = {
    'Gross debt - General government - Percent of GDP': 'Debt_GDP',
    'Current account balance (credit less debit) - Percent of GDP': 'CA_GDP',
    'Gross domestic product (GDP) - Current prices - Per capita - US dollar': 'GDP_pc',
    'Net lending (+) / net borrowing (-) - General government - Percent of GDP': 'Fiscal_Balance_GDP',
    'Gross national savings - Percent of GDP': 'Savings_GDP',
    'External debt - Percent of GDP': 'External_Debt_GDP',
}

def extract_variable(df, indicator_name, short_name):
    filtered = df[df['INDICATOR'] == indicator_name].drop(columns=['INDICATOR', 'FREQUENCY', 'SCALE'])
    filtered.set_index('COUNTRY', inplace=True)
    
    # Filter to selected countries
    available_countries = [c for c in all_countries if c in filtered.index]
    filtered = filtered.loc[filtered.index.isin(available_countries)]
    
    # Transpose: index = years, columns = countries
    transposed = filtered.T
    transposed.index = transposed.index.astype(int)
    transposed.columns.name = None
    
    return transposed

data_dict = {}
for indicator, short_name in variables_of_interest.items():
    try:
        data_dict[short_name] = extract_variable(IMFWEO_data, indicator, short_name)
        print(f"Extracted {short_name}: {data_dict[short_name].shape}")
    except Exception as e:
        print(f"Failed {short_name}: {e}")

In [ ]:
year_range = range(2014, 2025)
plot_vars = ['Debt_GDP', 'CA_GDP', 'GDP_pc', 'Fiscal_Balance_GDP', 'Savings_GDP', 'External_Debt_GDP']
var_labels = {
    'Debt_GDP': 'Government Debt (% GDP)',
    'CA_GDP': 'Current Account (% GDP)', 
    'GDP_pc': 'GDP per Capita (USD)',
    'Fiscal_Balance_GDP': 'Fiscal Balance (% GDP)',
    'Savings_GDP': 'Gross National Savings (% GDP)',
    'External_Debt_GDP': 'External Debt (% GDP)',
}


In [ ]:
def compute_group_avg(df, countries, years):
    available = [c for c in countries if c in df.columns]
    subset = df.loc[df.index.isin(years), available]
    return subset.mean(axis=1)
def get_benchmark(df, benchmark_name, years):
    if benchmark_name in df.columns:
        return df.loc[df.index.isin(years), benchmark_name]
    return pd.Series(dtype=float)


# Create comparison plots
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, var in enumerate(plot_vars):
    if var not in data_dict:
        axes[i].text(0.5, 0.5, f'{var}\nNot Available', ha='center', va='center')
        continue
    
    df = data_dict[var]
    
    # Compute group averages
    exp_avg = compute_group_avg(df, oil_exporters, year_range)
    ctrl_avg = compute_group_avg(df, controls, year_range)
    
    # Get benchmarks
    em_benchmark = get_benchmark(df, 'Emerging Market and Developing Economies', year_range)
    world_benchmark = get_benchmark(df, 'World', year_range)
    
    ax = axes[i]
    
    # Plot main groups (thick lines)
    ax.plot(exp_avg.index, exp_avg.values, 'b-o', label='Oil Exporters', 
            linewidth=2.5, markersize=5, zorder=5)
    ax.plot(ctrl_avg.index, ctrl_avg.values, 'r-s', label='Controls', 
            linewidth=2.5, markersize=5, zorder=5)
    
    # Plot benchmarks (thinner, dashed lines)
    if len(em_benchmark) > 0:
        ax.plot(em_benchmark.index, em_benchmark.values, 'g--', label='EM & Developing', 
                linewidth=1.5, alpha=0.7, zorder=3)
    if len(world_benchmark) > 0:
        ax.plot(world_benchmark.index, world_benchmark.values, 'k:', label='World', 
                linewidth=1.5, alpha=0.7, zorder=3)
    
    ax.set_title(var_labels.get(var, var), fontsize=12, fontweight='bold')
    ax.set_xlabel('Year')
    ax.legend(fontsize=8, loc='best')
    ax.grid(True, alpha=0.3)
    
    # Rotate x-axis labels
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Oil Exporters vs Controls vs Global Benchmarks\n(Group Averages, Key CDS Determinants)', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('group_comparison_with_benchmarks.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
all_countries = oil_exporters + controls

# Variables
variables = {
    'Gross debt - General government - Percent of GDP': 'Debt/GDP',
    'External debt - Percent of GDP': 'Ext Debt/GDP',
    'Current account balance (credit less debit) - Percent of GDP': 'CA/GDP',
    'Gross domestic product (GDP) - Current prices - Per capita - US dollar': 'GDP pc',
    'Net lending (+) / net borrowing (-) - General government - Percent of GDP': 'Fiscal Bal',
    'Gross national savings - Percent of GDP': 'Savings/GDP',
}

# Years to average
year_range = range(2014, 2025)

# Build table
table_data = {}

for indicator, short_name in variables.items():
    filtered = IMFWEO_data[IMFWEO_data['INDICATOR'] == indicator].drop(columns=['INDICATOR', 'FREQUENCY', 'SCALE'])
    filtered.set_index('COUNTRY', inplace=True)
    filtered = filtered.loc[filtered.index.isin(all_countries)]
    transposed = filtered.T
    transposed.index = transposed.index.astype(int)
    
    # Average over years
    avg = transposed.loc[transposed.index.isin(year_range)].mean()
    table_data[short_name] = avg

# Combine into single DataFrame
result = pd.DataFrame(table_data)

# Add group column
result['Group'] = result.index.map(lambda x: 'Exporter' if x in oil_exporters else 'Control')

# Reorder columns
result = result[['Group', 'Debt/GDP', 'Ext Debt/GDP', 'CA/GDP', 'GDP pc', 'Fiscal Bal', 'Savings/GDP']]

# Sort by group then country
result = result.sort_values(['Fiscal Bal'], ascending=[True])

result

### Placebo classifications

Let's expand the 24 country list and build placebo classifications to see if oil effect persists.

In [ ]:
high_gdp_pc = ['Qatar','Abu Dhabi', 'Dubai','South Korea','Saudi Arabia','Chile','Turkey','Malaysia','Mexico','China']
low_gdp_pc = ['Philippines', 'Egypt','Indonesia','South Africa','Colombia','Brazil']

high_debt_to_gdp = ['Egypt','Brazil','China','Malaysia','South Africa','Colombia','Mexico','Qatar']
low_debt_to_gdp = ['Saudi Arabia','Abu Dhabi','Dubai','Chile','Turkey','Indonesia','South Korea','Philippines']

high_CA_to_GDP = ['Qatar','Abu Dhabi','Dubai','South Korea','Malaysia','Saudi Arabia', 'China','Philippines','Mexico']
low_CA_to_GDP = ['Colombia','Chile','Egypt','Brazil','Turkey','South Africa','Indonesia']

high_fiscal_bal = ['Qatar','Abu Dhabi','Dubai','South Korea']
low_fiscal_bal = ['Egypt','Brazil','Saudi Arabia','China','South Africa','Colombia','Mexico','Malaysia','Chile','Turkey','Indonesia','Philippines']

GCC_countries = ['Qatar','Abu Dhabi','Dubai','Saudi Arabia']
non_GCC_countries = ['Egypt','Brazil','South Korea','China','South Africa','Colombia','Mexico','Malaysia','Chile','Turkey','Indonesia','Philippines']

asian_countries = ['South Korea', 'Malaysia', 'China', 'Indonesia', 'Philippines', 'Thailand']
non_asian_countries = ['Qatar', 'Abu Dhabi', 'Dubai', 'Saudi Arabia', 'Egypt', 'Brazil', 'Colombia', 'Mexico', 'Chile', 'South Africa', 'Turkey']

latin_american_countries = ['Brazil', 'Colombia', 'Mexico', 'Chile']
non_latin_american_countries = ['Qatar', 'Abu Dhabi', 'Dubai', 'Saudi Arabia', 'Egypt', 'South Korea', 'Malaysia', 'China', 'Indonesia', 'Philippines', 'Thailand', 'South Africa', 'Turkey']

# Credit ratings (approximate, investment grade A- or better vs BBB+ or below)
high_credit_rating = ['Qatar', 'Abu Dhabi', 'Dubai', 'Saudi Arabia', 'South Korea', 'Chile', 'China', 'Malaysia']
low_credit_rating = ['Egypt', 'Brazil', 'Colombia', 'Mexico', 'Indonesia', 'Philippines', 'Turkey', 'South Africa', 'Thailand']

In [ ]:
CDS_data = pd.read_csv('../data/processed/CDS/Weekly_CDS.csv')
CDS_data['Date'] = pd.to_datetime(CDS_data['Date'])
CDS_data.set_index('Date', inplace=True)

Oil_data = pd.read_csv('../data/processed/Oil/oil_prices_datastream.csv')
Oil_data['Date'] = pd.to_datetime(Oil_data['Date'])
Oil_data.set_index('Date', inplace=True)

Macro_risk_variables = pd.read_csv('../data/processed/Macroeconomic_variables/macro_risk_variables.csv')
Macro_risk_variables['Date'] = pd.to_datetime(Macro_risk_variables['Date'])
Macro_risk_variables.set_index('Date', inplace=True)

VIX = pd.read_csv('../data/processed/Macroeconomic_variables/VIXCLS.csv')
VIX['Date'] = pd.to_datetime(VIX['Date'])
VIX.set_index('Date', inplace=True)

OVX = pd.read_csv('../data/processed/Macroeconomic_variables/OVXCLS.csv')
OVX['Date'] = pd.to_datetime(OVX['Date'])
OVX.set_index('Date', inplace=True)

# =============================================================================
# MERGE (WIDE FORMAT)
# =============================================================================

merged = CDS_data.copy()
merged = pd.merge(merged, Oil_data, left_index=True, right_index=True, how='inner')
merged = pd.merge(merged, Macro_risk_variables, left_index=True, right_index=True, how='inner')
merged = pd.merge(merged, VIX, left_index=True, right_index=True, how='inner')
merged = pd.merge(merged, OVX, left_index=True, right_index=True, how='inner')

# Filter date range
merged = merged[merged.index >= datetime.datetime(2014, 1, 1)].copy()
merged.reset_index(inplace=True)

# Oil returns (percent change)
merged['Oil_ret'] = merged['Brent'].pct_change()

# OVX change (percent change - it's a volatility index)
merged['OVX_chg'] = merged['OVXCLS'].pct_change()

# VIX change (percent change)
merged['VIX_chg'] = merged['VIXCLS'].pct_change()

# DXY change (percent change)
merged['DXY_chg'] = merged['DXY'].pct_change()

# Treasury change (diff change)
merged['UST2Y_chg'] = merged['UST2Y'].diff()
merged['UST5Y_chg'] = merged['UST5Y'].diff()
merged['UST10Y_chg'] = merged['UST10Y'].diff()


# Drop NA values from merged
#merged = merged.dropna().reset_index(drop=True)

# Check dimensions and range

print(f"\nMerged shape: {merged.shape}")
print(f"Date range: {merged['Date'].min()} to {merged['Date'].max()}")


# =============================================================================
# RESHAPE TO LONG FORMAT
# =============================================================================

oil_exporters = ['Saudi Arabia', 'Abu Dhabi', 'Dubai', 'Qatar', 'Colombia', 
                 'Mexico', 'Brazil', 'Egypt', 'Malaysia']
controls = ['Indonesia', 'Philippines', 'Turkey', 'Chile', 'China', 
            'South Africa', 'South Korea', 'Thailand']


global_vars = ['Date', 'Brent', 'Oil_ret', 'OVXCLS', 'OVX_chg', 
               'VIXCLS', 'VIX_chg', 'DXY_chg', 'UST2Y_chg', 'UST5Y_chg', 'UST10Y_chg']

merged = merged.reset_index()

panel_data = []

for country in oil_exporters + controls:
    if country not in merged.columns:
        print(f"Warning: {country} not in CDS data")
        continue
        
    temp = merged[global_vars + [country]].copy()
    temp['Country'] = country
    temp['CDS'] = temp[country]
    temp['OilExporter'] = 1 if country in oil_exporters else 0
    temp = temp.drop(columns=[country])
    panel_data.append(temp)

panel = pd.concat(panel_data, ignore_index=True)
panel = panel.sort_values(by=['Country', 'Date']).reset_index(drop=True)

# =============================================================================
# COMPUTE COUNTRY-SPECIFIC VARIABLES (CDS returns)
# =============================================================================

# CDS returns - THIS one needs groupby because it's country-specific
panel['CDS_ret'] = panel.groupby('Country')['CDS'].pct_change()

# Create interaction term
panel['OVX_x_Exporter'] = panel['OVX_chg'] * panel['OilExporter']
panel['Brent_chg_x_Exporter'] = panel['Oil_ret'] * panel['OilExporter']

# Drop NaN (first observation per country)
panel = panel.dropna(subset=['CDS_ret', 'OVX_chg', 'VIX_chg', 'DXY_chg', 'UST2Y_chg', 'UST5Y_chg', 'UST10Y_chg']).reset_index(drop=True)

panel.head()

In [ ]:
from linearmodels.panel import PanelOLS
import pandas as pd
import numpy as np

# =============================================================================
# DEFINE ALL CLASSIFICATIONS
# =============================================================================

classifications = {
    'Oil Exporters': {
        'treatment': ['Saudi Arabia', 'Abu Dhabi', 'Dubai', 'Qatar', 'Colombia', 'Mexico', 'Brazil', 'Egypt', 'Malaysia'],
        'control': ['Indonesia', 'Philippines', 'Turkey', 'Chile', 'China', 'South Africa', 'South Korea', 'Thailand']
    },
    'GCC': {
        'treatment': ['Qatar', 'Abu Dhabi', 'Dubai', 'Saudi Arabia'],
        'control': ['Egypt', 'Brazil', 'South Korea', 'China', 'South Africa', 'Colombia', 'Mexico', 'Malaysia', 'Chile', 'Turkey', 'Indonesia', 'Philippines', 'Thailand']
    },
    'Low GDP pc': {
        'control': ['Qatar', 'Abu Dhabi', 'Dubai', 'South Korea', 'Saudi Arabia', 'Chile', 'Turkey', 'Malaysia', 'Mexico', 'China'],
        'treatment': ['Philippines', 'Egypt', 'Indonesia', 'South Africa', 'Colombia', 'Brazil', 'Thailand']
    },
    'High Debt/GDP': {
        'treatment': ['Egypt', 'Brazil', 'China', 'Malaysia', 'South Africa', 'Colombia', 'Mexico', 'Qatar'],
        'control': ['Saudi Arabia', 'Abu Dhabi', 'Dubai', 'Chile', 'Turkey', 'Indonesia', 'South Korea', 'Philippines', 'Thailand']
    },
    'Low CA/GDP': {
        'control': ['Qatar', 'Abu Dhabi', 'Dubai', 'South Korea', 'Malaysia', 'Saudi Arabia', 'China', 'Philippines', 'Mexico'],
        'treatment': ['Colombia', 'Chile', 'Egypt', 'Brazil', 'Turkey', 'South Africa', 'Indonesia', 'Thailand']
    },
    'Low Fiscal Bal': {
        'control': ['Qatar', 'Abu Dhabi', 'Dubai', 'South Korea'],
        'treatment': ['Egypt', 'Brazil', 'Saudi Arabia', 'China', 'South Africa', 'Colombia', 'Mexico', 'Malaysia', 'Chile', 'Turkey', 'Indonesia', 'Philippines', 'Thailand']
    },
    'Asian': {
        'treatment': ['South Korea', 'Malaysia', 'China', 'Indonesia', 'Philippines', 'Thailand'],
        'control': ['Qatar', 'Abu Dhabi', 'Dubai', 'Saudi Arabia', 'Egypt', 'Brazil', 'Colombia', 'Mexico', 'Chile', 'South Africa', 'Turkey']
    },
    'Latin America': {
        'treatment': ['Brazil', 'Colombia', 'Mexico', 'Chile'],
        'control': ['Qatar', 'Abu Dhabi', 'Dubai', 'Saudi Arabia', 'Egypt', 'South Korea', 'Malaysia', 'China', 'Indonesia', 'Philippines', 'Thailand', 'South Africa', 'Turkey']
    },
    'Low Credit Rating': {
        'control': ['Qatar', 'Abu Dhabi', 'Dubai', 'Saudi Arabia', 'South Korea', 'Chile', 'China', 'Malaysia'],
        'treatment': ['Egypt', 'Brazil', 'Colombia', 'Mexico', 'Indonesia', 'Philippines', 'Turkey', 'South Africa', 'Thailand']
    },
}

# Thresholds to test
thresholds = [1.0, 0.10, 0.05, 0.01]  # 1.0 = full sample

# =============================================================================
# RUN ALL REGRESSIONS
# =============================================================================

results_list = []

for class_name, groups in classifications.items():
    
    treatment = groups['treatment']
    control = groups['control']
    
    # Create classification-specific panel
    temp_panel = panel.copy()
    temp_panel['Treatment'] = temp_panel['Country'].isin(treatment).astype(int)
    temp_panel['OVX_x_Treatment'] = temp_panel['OVX_chg'] * temp_panel['Treatment']
    
    for pct in thresholds:
        
        # Filter to threshold
        if pct == 1.0:
            subset = temp_panel.copy()
            threshold_label = 'Full'
        else:
            oil_qtl = temp_panel['OVX_chg'].quantile(1 - pct)
            subset = temp_panel[temp_panel['OVX_chg'] > oil_qtl].copy()
            threshold_label = f'Top {pct*100:.0f}%'
        
        # Skip if too few observations
        if len(subset) < 50:
            continue
        
        # Prepare for linearmodels
        subset['Date'] = pd.to_datetime(subset['Date'])
        reg_data = subset.set_index(['Country', 'Date'])
        
        y = reg_data['CDS_ret']
        
        # Model A: Country FE + explicit controls
        try:
            X_a = reg_data[['OVX_chg', 'OVX_x_Treatment', 'VIX_chg', 'DXY_chg', 'UST10Y_chg']]
            model_a = PanelOLS(y, X_a, entity_effects=True, time_effects=False, drop_absorbed=True)
            res_a = model_a.fit(cov_type='clustered', cluster_entity=True)
            beta_a = res_a.params['OVX_x_Treatment']
            pval_a = res_a.pvalues['OVX_x_Treatment']
        except:
            beta_a, pval_a = np.nan, np.nan
        
        # Model B: Country FE + Time FE
        try:
            X_b = reg_data[['OVX_x_Treatment']]
            model_b = PanelOLS(y, X_b, entity_effects=True, time_effects=True, drop_absorbed=True)
            res_b = model_b.fit(cov_type='clustered', cluster_entity=True)
            beta_b = res_b.params['OVX_x_Treatment']
            pval_b = res_b.pvalues['OVX_x_Treatment']
        except:
            beta_b, pval_b = np.nan, np.nan
        
        results_list.append({
            'Classification': class_name,
            'Threshold': threshold_label,
            'N': len(subset),
            'N_treatment': subset['Treatment'].sum(),
            'N_control': len(subset) - subset['Treatment'].sum(),
            'Beta_A': beta_a,
            'pval_A': pval_a,
            'Beta_B': beta_b,
            'pval_B': pval_b,
        })

# =============================================================================
# CREATE RESULTS DATAFRAME
# =============================================================================

results_df = pd.DataFrame(results_list)

# =============================================================================
# PRINT RESULTS - MODEL A
# =============================================================================

print("="*90)
print("MODEL A: Country FE + Global Controls (VIX, DXY, UST10Y)")
print("="*90)

# Pivot for easier reading
pivot_a = results_df.pivot(index='Classification', columns='Threshold', values='Beta_A')
pivot_a = pivot_a[['Full', 'Top 10%', 'Top 5%', 'Top 1%']]  # Order columns

pivot_pval_a = results_df.pivot(index='Classification', columns='Threshold', values='pval_A')
pivot_pval_a = pivot_pval_a[['Full', 'Top 10%', 'Top 5%', 'Top 1%']]

# Print with significance stars
print(f"\n{'Classification':<20} {'Full':>12} {'Top 10%':>12} {'Top 5%':>12} {'Top 1%':>12}")
print("-"*70)

for idx in pivot_a.index:
    row_str = f"{idx:<20}"
    for col in pivot_a.columns:
        beta = pivot_a.loc[idx, col]
        pval = pivot_pval_a.loc[idx, col]
        
        if pd.isna(beta):
            row_str += f"{'N/A':>12}"
        else:
            sig = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''
            row_str += f"{beta:>10.3f}{sig:<2}"
    print(row_str)

print("\n* p<0.1, ** p<0.05, *** p<0.01")

# =============================================================================
# PRINT RESULTS - MODEL B
# =============================================================================

print("\n" + "="*90)
print("MODEL B: Country FE + Time FE")
print("="*90)

pivot_b = results_df.pivot(index='Classification', columns='Threshold', values='Beta_B')
pivot_b = pivot_b[['Full', 'Top 10%', 'Top 5%', 'Top 1%']]

pivot_pval_b = results_df.pivot(index='Classification', columns='Threshold', values='pval_B')
pivot_pval_b = pivot_pval_b[['Full', 'Top 10%', 'Top 5%', 'Top 1%']]

print(f"\n{'Classification':<20} {'Full':>12} {'Top 10%':>12} {'Top 5%':>12} {'Top 1%':>12}")
print("-"*70)

for idx in pivot_b.index:
    row_str = f"{idx:<20}"
    for col in pivot_b.columns:
        beta = pivot_b.loc[idx, col]
        pval = pivot_pval_b.loc[idx, col]
        
        if pd.isna(beta):
            row_str += f"{'N/A':>12}"
        else:
            sig = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''
            row_str += f"{beta:>10.3f}{sig:<2}"
    print(row_str)

print("\n* p<0.1, ** p<0.05, *** p<0.01")

# =============================================================================
# HIGHLIGHT: COMPARE OIL EXPORTERS VS OTHER CLASSIFICATIONS
# =============================================================================

print("\n" + "="*90)
print("COMPARISON: Oil Exporters vs Placebo Classifications (Top 10% OVX, Model A)")
print("="*90)

top10_results = results_df[results_df['Threshold'] == 'Top 10%'].copy()
top10_results = top10_results.sort_values('Beta_A', ascending=False)

print(f"\n{'Classification':<20} {'Beta':>10} {'p-value':>10} {'Significant':>12}")
print("-"*55)

for _, row in top10_results.iterrows():
    sig = '***' if row['pval_A'] < 0.01 else '**' if row['pval_A'] < 0.05 else '*' if row['pval_A'] < 0.1 else ''
    is_sig = 'Yes' if row['pval_A'] < 0.05 else 'No'
    print(f"{row['Classification']:<20} {row['Beta_A']:>10.3f} {row['pval_A']:>10.3f} {is_sig:>10} {sig}")

---

# 2. Cross-Commodity Correlations

Correlations of Brent against other commodities (cash levels and 12-month futures), used to
motivate Brent as the primary energy benchmark.


In [ ]:
import pandas as pd


In [ ]:
oil_futures = pd.read_csv('../data/processed/Oil/oil_futures.csv')
oil_futures['date'] = pd.to_datetime(oil_futures['date'], format='%d.%m.%Y')
oil_futures = oil_futures.sort_values('date')


oil_prices = pd.read_csv('../data/processed/Oil/oil_prices_datastream.csv').sort_values('date')
oil_prices['date'] = pd.to_datetime(oil_prices['date'], format='%m/%d/%y')
oil_prices = oil_prices.sort_values('date')


commodities = pd.read_csv('../data/processed/Macroeconomic_variables/other_commodities.csv')
commodities['date'] = pd.to_datetime(commodities['date'], format='%d.%m.%Y')
commodities = commodities.sort_values('date')


df = pd.merge_asof(
    oil_prices[['date','Brent']], 
    oil_futures[['date', 'Brent_12m']], 
    on='date', 
    direction='nearest'
)

df = pd.merge_asof(
    df, 
    commodities,
    on='date', 
    direction='nearest'
)

df.rename(columns={'Brent':'Brent_cash'}, inplace=True)
df

In [ ]:
import numpy as np
import pandas as pd

commodities = ['Copper', 'Nickel', 'CokingCoal', 'ThermalCoal', 'Platinum', 'IronOre', 'NatGas']

# Restrict to the thesis sample period if desired
# df_s = df[(df['date'] >= '2015-01-01') & (df['date'] <= '2024-12-31')].copy()
df_s = df.copy()  # or use the full sample, your choice

# Correlations of cash/spot prices with Brent cash
print("\n\nCorrelation of futures prices with Brent futures\n")
focus_cash = pd.DataFrame({
    'Levels':  df_s[[f'{c}_12m' for c in ['Brent'] + commodities]].corr()['Brent_12m'],
}).round(3).drop('Brent_12m')
print(focus_cash.to_string())

---

# 3. Empirical Characterization

Three blocks: imports & panel build, OVX-conditioned regressions, and futures-basis regressions.


### 0. Imports & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import datetime
import warnings
warnings.filterwarnings('ignore')

from linearmodels.panel import PanelOLS
from scipy.optimize import minimize
from scipy.stats import chi2, ttest_ind, mannwhitneyu, fisher_exact
from numba import njit

In [ ]:
# ── Country groups ──────────────────────────────────────────
oil_exporters = [
    'Saudi Arabia', 'Abu Dhabi', 'Qatar', 'Colombia',
    'Mexico', 'Brazil', 'Egypt', 'Malaysia'
]
control_countries = [
    'Indonesia', 'Philippines', 'Turkey', 'Chile', 'China',
    'South Africa', 'South Korea', 'Thailand'
]
all_countries = oil_exporters + control_countries

In [ ]:
# ── Load datasets ──────────────────────────────────────────
CDS_data  = pd.read_csv('../data/processed/CDS/Weekly_CDS.csv', parse_dates=['date'], index_col='date')
Oil_data  = pd.read_csv('../data/processed/Oil/oil_prices_datastream.csv', parse_dates=['date'], index_col='date')
Oil_fut   = pd.read_csv('../data/processed/Oil/oil_futures.csv', parse_dates=['date'], index_col='date')
Macro     = pd.read_csv('../data/processed/Macroeconomic_variables/macro_risk_variables.csv', parse_dates=['Date'], index_col='Date')
VIX       = pd.read_csv('../data/processed/Macroeconomic_variables/VIXCLS.csv', parse_dates=['Date'], index_col='Date')
OVX       = pd.read_csv('../data/processed/Macroeconomic_variables/OVXCLS.csv', parse_dates=['date'], index_col='date')
FX_data   = pd.read_csv('../data/processed/Macroeconomic_variables/Daily_FX_Rates.csv', parse_dates=['Date'], index_col='Date')
MSCI_data = pd.read_csv('../data/processed/MSCI_indices/mscicountryindex.csv', parse_dates=['date'], index_col='date')

print(f'CDS: {CDS_data.shape}, Oil: {Oil_data.shape}, Futures: {Oil_fut.shape}')

### 0.1 Build Panel

In [ ]:
# ── Merge global variables ─────────────────────────────────
merged = (
    CDS_data
    .join(Oil_data, how='inner')
    .join(Oil_fut, how='inner')
    .join(Macro, how='inner')
    .join(VIX, how='inner')
    .join(OVX, how='inner')
)
merged = merged.loc[(merged.index >= '2014-01-01') & (merged.index <= '2024-12-31')].copy()

# ── Global transforms ──────────────────────────────────────
merged['Oil_ret']    = np.log(merged['Brent'] / merged['Brent'].shift(1))
merged['OVX_chg']    = np.log(merged['OVXCLS'] / merged['OVXCLS'].shift(1))
merged['VIX_chg']    = np.log(merged['VIXCLS'] / merged['VIXCLS'].shift(1))
merged['DXY_chg']    = np.log(merged['DXY'] / merged['DXY'].shift(1))
merged['UST10Y_chg'] = merged['UST10Y'].diff()

# ── Futures curve ratios ───────────────────────────────────
for tenor, col in [('1m','Brent_1m'),('3m','Brent_3m'),('6m','Brent_6m'),
                    ('12m','Brent_12m'),('24m','Brent_24m')]:
    if col in merged.columns:
        merged[f'log_curve_{tenor}']   = np.log(merged[col] / merged['Brent'])
        merged[f'd_log_curve_{tenor}'] = merged[f'log_curve_{tenor}'].diff()

# ── Reshape to long panel ──────────────────────────────────
global_cols = [c for c in merged.columns if c not in all_countries]
merged = merged.reset_index()

fx_aligned   = FX_data.reindex(pd.to_datetime(merged['date'])).reset_index()
msci_aligned = MSCI_data.reindex(pd.to_datetime(merged['date'])).reset_index()

panel_rows = []
for country in all_countries:
    if country not in merged.columns:
        print(f'Warning: {country} not in CDS data')
        continue
    temp = merged[['date'] + global_cols + [country]].copy() if country in merged.columns else None
    # Simpler: just grab global + CDS
    temp = merged[['date'] + [c for c in global_cols if c in merged.columns]].copy()
    temp['CDS'] = merged[country].values
    temp['Country'] = country
    temp['OilExporter'] = int(country in oil_exporters)
    if country in fx_aligned.columns:
        temp['FX'] = fx_aligned[country].values
    else:
        temp['FX'] = np.nan
    if country in msci_aligned.columns:
        temp['MSCI'] = msci_aligned[country].values
    else:
        temp['MSCI'] = np.nan
    panel_rows.append(temp)

panel = pd.concat(panel_rows, ignore_index=True).sort_values(['Country','date']).reset_index(drop=True)

# ── Country-specific log-returns ───────────────────────────
panel['CDS_ret']  = panel.groupby('Country')['CDS'].transform(lambda s: np.log(s / s.shift(1)))
panel['FX_ret']   = panel.groupby('Country')['FX'].transform(lambda s: np.log(s / s.shift(1)))
panel['MSCI_ret'] = panel.groupby('Country')['MSCI'].transform(lambda s: np.log(s / s.shift(1)))

# ── Interaction terms ──────────────────────────────────────
panel['Brent_x_Exporter'] = panel['Oil_ret'] * panel['OilExporter']

# ── Drop incomplete rows ──────────────────────────────────
panel = panel.dropna(subset=['CDS_ret','Oil_ret','VIX_chg','DXY_chg','UST10Y_chg']).reset_index(drop=True)

print(f'Panel: {panel.shape[0]} rows, {panel["Country"].nunique()} countries, '
      f'~{len(panel)//panel["Country"].nunique()} weeks each')

---

### 1. OVX-Conditioned Regressions

### 1.1 Panel regressions: Differential oil sensitivity

Four nested specifications testing whether $\beta_2$ on `Brent × Exporter` is negative and significant.

In [ ]:
reg_panel = panel.copy()
reg_panel = reg_panel.set_index(['Country', 'date'])
y = reg_panel['CDS_ret']

In [ ]:
# Model 1: Baseline
X_1 = reg_panel[['Oil_ret', 'Brent_x_Exporter']]
results_1 = PanelOLS(y, X_1, entity_effects=False, time_effects=False
            ).fit(cov_type='clustered', cluster_entity=True)

# Model 2: + controls
X_2 = reg_panel[['Oil_ret', 'Brent_x_Exporter', 'VIX_chg',
                 'DXY_chg', 'MSCI_ret', 'FX_ret', 'UST10Y_chg']]
results_2 = PanelOLS(y, X_2, entity_effects=False, time_effects=False
            ).fit(cov_type='clustered', cluster_entity=True)

# Model 3: + country FE
X_3 = reg_panel[['Oil_ret', 'Brent_x_Exporter', 'VIX_chg',
                 'DXY_chg', 'MSCI_ret', 'FX_ret', 'UST10Y_chg']]
results_3 = PanelOLS(y, X_3, entity_effects=True, time_effects=False
            ).fit(cov_type='clustered', cluster_entity=True)

# Model 4: + country FE + time FE
X_4 = reg_panel[['Brent_x_Exporter']]
results_4 = PanelOLS(y, X_4, entity_effects=True, time_effects=True
            ).fit(cov_type='clustered', cluster_entity=True)

In [ ]:
# ── Summary table ─────────────────────────────────────────
print(f"\n{'Model':<45} {'β':>10} {'SE':>10} {'p-value':>10} {'R²':>10}")
print('-' * 85)

for name, res in [('Model 1: Baseline', results_1),
                   ('Model 2: + Global Controls', results_2),
                   ('Model 3: + Country FE', results_3),
                   ('Model 4: + Country FE + Time FE', results_4)]:
    beta = res.params['Brent_x_Exporter']
    se   = res.std_errors['Brent_x_Exporter']
    pval = res.pvalues['Brent_x_Exporter']
    r2   = res.rsquared_within if hasattr(res, 'rsquared_within') else res.rsquared
    sig  = '***' if pval<0.01 else '**' if pval<0.05 else '*' if pval<0.1 else ''
    print(f'{name:<45} {beta:>10.4f} {se:>10.4f} {pval:>10.4f} {r2:>10.4f} {sig}')

print('\n* p<0.1, ** p<0.05, *** p<0.01 — SEs clustered by country')

### 1.2 OVX threshold regressions

Restrict sample to weeks when OVX > quantile, re-estimate. If the exporter differential is tail-driven, $|\hat{\beta}_2|$ should increase.

In [ ]:
results_list = []

ovx_quantiles = {
    'Full Sample': None,
    'OVX > Q70':   panel['OVXCLS'].quantile(0.70),
    'OVX > Q80':   panel['OVXCLS'].quantile(0.80),
    'OVX > Q90':   panel['OVXCLS'].quantile(0.90),
    'OVX > Q95':   panel['OVXCLS'].quantile(0.95),
    'OVX > Q99':   panel['OVXCLS'].quantile(0.99),
}

for label, cutoff in ovx_quantiles.items():
    subset = panel.copy() if cutoff is None else panel[panel['OVXCLS'] > cutoff].copy()
    subset = subset.set_index(['Country', 'date'])

    y_sub = subset['CDS_ret']
    X = subset[['Oil_ret', 'Brent_x_Exporter', 'VIX_chg',
                'DXY_chg', 'MSCI_ret', 'FX_ret', 'UST10Y_chg']]
    res = PanelOLS(y_sub, X, entity_effects=True, time_effects=False,
                   drop_absorbed=True, check_rank=False
          ).fit(cov_type='clustered', cluster_entity=True)

    results_list.append({
        'Threshold': label, 'N': len(y_sub),
        'β': res.params['Brent_x_Exporter'],
        'SE': res.std_errors['Brent_x_Exporter'],
        'p': res.pvalues['Brent_x_Exporter'],
    })

print(f"{'Threshold':<15} {'N':>6}  {'β':>8} {'p':>7}")
print('-' * 40)
for r in results_list:
    sig = '***' if r['p']<0.01 else '**' if r['p']<0.05 else '*' if r['p']<0.1 else ''
    print(f"{r['Threshold']:<15} {r['N']:>6}  {r['β']:>8.4f} {r['p']:>6.3f}{sig}")

---

### 2. Futures Basis Regressions

Test whether the oil futures term structure (log curve ratio at various tenors) affects CDS changes differently for exporters vs. controls.

### 2.1 Interaction specification (changes in curve ratio)

In [ ]:
controls_list = ['VIX_chg', 'FX_ret', 'MSCI_ret', 'UST10Y_chg', 'DXY_chg']

for tenor, tlabel in [('d_log_curve_1m','1m'),('d_log_curve_3m','3m'),
                       ('d_log_curve_6m','6m'),('d_log_curve_12m','12m'),
                       ('d_log_curve_24m','24m')]:
    if tenor not in panel.columns:
        continue
    panel[f'Curve_{tlabel}_x_Exp'] = panel[tenor] * panel['OilExporter']

    ivs = [tenor, f'Curve_{tlabel}_x_Exp'] + controls_list
    sub = panel.dropna(subset=['CDS_ret'] + ivs).copy()
    sub = sub.set_index(['Country', 'date'])

    mod = PanelOLS(sub['CDS_ret'], sub[ivs], entity_effects=True
          ).fit(cov_type='clustered', cluster_entity=True)

    b_base = mod.params[tenor]
    p_base = mod.pvalues[tenor]
    b_int  = mod.params[f'Curve_{tlabel}_x_Exp']
    p_int  = mod.pvalues[f'Curve_{tlabel}_x_Exp']

    sb = '***' if p_base<0.01 else '**' if p_base<0.05 else '*' if p_base<0.1 else ''
    si = '***' if p_int<0.01  else '**' if p_int<0.05  else '*' if p_int<0.1  else ''

    print(f'Tenor {tlabel}:  Base β={b_base:.3f}{sb}  Interaction β={b_int:.3f}{si}  '
          f'Total(exp)={b_base+b_int:.3f}  R²={mod.rsquared:.4f}  N={int(mod.nobs)}')

### 2.2 Split-sample: exporters vs. controls separately

In [ ]:
for tenor, tlabel in [('d_log_curve_1m','1m'),('d_log_curve_3m','3m'),
                       ('d_log_curve_6m','6m'),('d_log_curve_12m','12m'),
                       ('d_log_curve_24m','24m')]:
    if tenor not in panel.columns:
        continue
    ivs = [tenor] + controls_list

    print(f'\nTenor: {tlabel}')
    for group, glabel in [(1, 'Exporters'), (0, 'Controls')]:
        sub = panel[panel['OilExporter'] == group].dropna(subset=['CDS_ret'] + ivs).copy()
        sub = sub.set_index(['Country', 'date'])
        mod = PanelOLS(sub['CDS_ret'], sub[ivs], entity_effects=True
              ).fit(cov_type='clustered', cluster_entity=True)
        b = mod.params[tenor]
        p = mod.pvalues[tenor]
        sig = '***' if p<0.01 else '**' if p<0.05 else '*' if p<0.1 else ''
        print(f'  {glabel:<12} β={b:>7.3f}{sig:<3} p={p:.4f}  R²={mod.rsquared:.4f}  N={int(mod.nobs)}')

### 2.3 Level of curve ratio (split-sample)

In [ ]:
for tenor, tlabel in [('log_curve_1m','1m'),('log_curve_3m','3m'),
                       ('log_curve_6m','6m'),('log_curve_12m','12m'),
                       ('log_curve_24m','24m')]:
    if tenor not in panel.columns:
        continue
    ivs = [tenor] + controls_list

    print(f'\nTenor: {tlabel}')
    for group, glabel in [(1, 'Exporters'), (0, 'Controls')]:
        sub = panel[panel['OilExporter'] == group].dropna(subset=['CDS_ret'] + ivs).copy()
        sub = sub.set_index(['Country', 'date'])
        mod = PanelOLS(sub['CDS_ret'], sub[ivs], entity_effects=True
              ).fit(cov_type='clustered', cluster_entity=True)
        b = mod.params[tenor]
        p = mod.pvalues[tenor]
        sig = '***' if p<0.01 else '**' if p<0.05 else '*' if p<0.1 else ''
        print(f'  {glabel:<12} β={b:>7.3f}{sig:<3} p={p:.4f}  R²={mod.rsquared:.4f}  N={int(mod.nobs)}')

---

# 4. Jump Characterization & Estimation

## 4.1 GARCH vs. GARCH-Jump LR tests

Likelihood-ratio tests of GARCH(1,1) against a GARCH(1,1)-Jump alternative on oil benchmarks,
sovereign CDS and MSCI equity indices, followed by a co-jump analysis between Brent and CDS.


---

### 3. Jump Characterization

GARCH(1,1) vs. GARCH(1,1)-Jump likelihood ratio tests. The jump model nests the plain GARCH (set λ=0), so the LR statistic is χ²(3).

### 3.0 Estimation functions

In [ ]:
@njit
def garch_loglik_numba(params, returns):
    """GARCH(1,1) log-likelihood."""
    mu, omega, alpha, beta = params
    T = len(returns)
    h = np.zeros(T)
    h[0] = np.var(returns)
    for t in range(1, T):
        h[t] = omega + alpha * (returns[t-1] - mu)**2 + beta * h[t-1]
    ll = 0.0
    for t in range(T):
        ll += -0.5 * (np.log(2 * np.pi * h[t]) + (returns[t] - mu)**2 / h[t])
    return -ll


def estimate_garch(returns):
    x0 = [np.mean(returns), 1e-5, 0.05, 0.90]
    bounds = [(None, None), (1e-8, None), (1e-6, 1.0), (1e-4, 0.999)]
    result = minimize(garch_loglik_numba, x0, args=(returns,), method='L-BFGS-B',
                      bounds=bounds, options={'maxiter': 2000})
    return {'params': result.x, 'loglik': -result.fun, 'converged': result.success}


@njit
def garch_jump_loglik_numba(params, returns, max_jumps=10):
    """GARCH(1,1)-Jump log-likelihood with Poisson jumps."""
    mu, omega, alpha, beta, lam, theta, delta = params
    T = len(returns)
    h = np.zeros(T)
    h[0] = np.var(returns)
    for t in range(1, T):
        h[t] = omega + alpha * (returns[t-1] - mu)**2 + beta * h[t-1]
    ll = 0.0
    for t in range(T):
        prob_t = 0.0
        log_poisson = -lam
        for k in range(max_jumps + 1):
            var_k = h[t] + k * delta**2
            mean_k = mu + k * theta
            pdf_k = np.exp(-0.5 * (returns[t] - mean_k)**2 / var_k) / np.sqrt(2 * np.pi * var_k)
            poisson_k = np.exp(log_poisson)
            prob_t += pdf_k * poisson_k
            log_poisson += np.log(lam) - np.log(k + 1)
        ll += np.log(prob_t + 1e-10)
    return -ll


def estimate_garch_jump(returns):
    x0 = [np.mean(returns), 1e-5, 0.05, 0.85, 0.03, -0.01, 0.03]
    bounds = [(None, None), (1e-8, None), (1e-6, 1.0), (1e-4, 0.999),
              (1e-4, 1.5), (-0.2, 0.2), (1e-4, 0.5)]
    result = minimize(garch_jump_loglik_numba, x0, args=(returns,), method='L-BFGS-B',
                      bounds=bounds, options={'maxiter': 2000})
    return {'params': result.x, 'loglik': -result.fun, 'converged': result.success}

### 3.1 Load daily data for jump estimation

In [ ]:
# Daily data for jump tests (higher frequency = more power)
CDS_daily  = pd.read_csv('../data/processed/CDS/Daily_CDS.csv', parse_dates=['date'], index_col='date')
Oil_daily  = pd.read_csv('../data/processed/Oil/oil_prices_datastream.csv', parse_dates=['date'], index_col='date')
MSCI_daily = pd.read_csv('../data/processed/MSCI_indices/mscicountryindex.csv', parse_dates=['date'], index_col='date')

# Filter 2014+
CDS_daily  = CDS_daily[CDS_daily.index >= '2014-01-01']
Oil_daily  = Oil_daily[Oil_daily.index >= '2014-01-01']
MSCI_daily = MSCI_daily[MSCI_daily.index >= '2014-01-01']

# Returns
oil_returns  = np.log(Oil_daily / Oil_daily.shift(1)).dropna()
cds_returns  = np.log(CDS_daily / CDS_daily.shift(1)).dropna()
msci_returns = np.log(MSCI_daily / MSCI_daily.shift(1)).dropna()

print(f'Oil: {oil_returns.shape}, CDS: {cds_returns.shape}, MSCI: {msci_returns.shape}')

In [ ]:
# Country lists for jump tests
jump_countries = [
    'Brazil', 'Chile', 'China', 'Colombia', 'Egypt',
    'Indonesia', 'South Korea', 'Malaysia', 'Mexico',
    'Philippines', 'Qatar', 'Saudi Arabia', 'South Africa',
    'Thailand', 'Turkey', 'Abu Dhabi', 'Dubai'
]

jump_oil_exporters = [
    'Saudi Arabia', 'Qatar', 'Abu Dhabi', 'Dubai',
    'Colombia', 'Mexico', 'Brazil', 'Malaysia', 'Egypt'
]

### 3.2 Oil benchmarks

In [ ]:
for benchmark in ['Brent', 'WTI', 'OPEC_basket', 'Dubai_Crude']:
    if benchmark not in Oil_daily.columns:
        continue
    oil_ret = Oil_daily[benchmark].pct_change().dropna().values

    garch     = estimate_garch(oil_ret)
    garch_jmp = estimate_garch_jump(oil_ret)

    T = len(oil_ret)
    LR = 2 * (garch_jmp['loglik'] - garch['loglik'])
    p  = 1 - chi2.cdf(LR, df=3)

    aic_g = -2*garch['loglik'] + 2*4
    aic_j = -2*garch_jmp['loglik'] + 2*7
    bic_g = -2*garch['loglik'] + 4*np.log(T)
    bic_j = -2*garch_jmp['loglik'] + 7*np.log(T)

    print(f'{benchmark}: LR={LR:.1f} p={p:.6f}  '
          f'AIC: {aic_g:.0f}→{aic_j:.0f}  BIC: {bic_g:.0f}→{bic_j:.0f}  '
          f'λ={garch_jmp["params"][4]:.3f} ({garch_jmp["params"][4]*252:.0f}/yr)  '
          f'θ={garch_jmp["params"][5]:.4f}  δ={garch_jmp["params"][6]:.4f}')

### 3.3 CDS series

In [ ]:
cds_jump_results = []

for country in jump_countries:
    if country not in cds_returns.columns:
        print(f'{country}: NOT FOUND')
        continue
    ret = cds_returns[country].dropna().values
    if len(ret) < 200:
        continue
    if (ret == 0).sum() / len(ret) > 0.3:
        continue
    if np.var(ret) < 1e-10:
        continue

    garch     = estimate_garch(ret)
    garch_jmp = estimate_garch_jump(ret)
    T = len(ret)
    LR = 2 * (garch_jmp['loglik'] - garch['loglik'])
    pval = 1 - chi2.cdf(LR, df=3)

    cds_jump_results.append({
        'Country': country, 'T': T,
        'LR': LR, 'p_value': pval,
        'AIC_GARCH': -2*garch['loglik'] + 2*4,
        'AIC_Jump':  -2*garch_jmp['loglik'] + 2*7,
        'BIC_GARCH': -2*garch['loglik'] + 4*np.log(T),
        'BIC_Jump':  -2*garch_jmp['loglik'] + 7*np.log(T),
        'alpha_GARCH': garch['params'][2],
        'alpha_Jump':  garch_jmp['params'][2],
        'alpha_drop':  garch['params'][2] - garch_jmp['params'][2],
        'lambda': garch_jmp['params'][4],
        'theta':  garch_jmp['params'][5],
        'delta':  garch_jmp['params'][6],
        'Oil_Exporter': country in jump_oil_exporters,
    })
    sig = '***' if pval<0.01 else '**' if pval<0.05 else '*' if pval<0.1 else ''
    print(f'{country}: LR={LR:.1f} p={pval:.4f}{sig} λ={garch_jmp["params"][4]:.3f}')

cds_jump_df = pd.DataFrame(cds_jump_results)
print('\n', cds_jump_df.to_string())

### 3.4 MSCI equity index series

In [ ]:
# For MSCI, use original names (UAE instead of Abu Dhabi/Dubai)
msci_country_list = [
    'Brazil', 'Chile', 'China', 'Colombia', 'Egypt',
    'Indonesia', 'South Korea', 'Malaysia', 'Mexico',
    'Philippines', 'Qatar', 'Saudi Arabia', 'South Africa',
    'Thailand', 'Turkey', 'United Arab Emirates'
]

msci_jump_results = []

for country in msci_country_list:
    if country not in msci_returns.columns:
        print(f'{country}: NOT FOUND')
        continue
    ret = msci_returns[country].dropna().values
    if len(ret) < 200:
        continue
    if (ret == 0).sum() / len(ret) > 0.3:
        continue

    garch     = estimate_garch(ret)
    garch_jmp = estimate_garch_jump(ret)
    T = len(ret)
    LR = 2 * (garch_jmp['loglik'] - garch['loglik'])
    pval = 1 - chi2.cdf(LR, df=3)

    is_oil = country in jump_oil_exporters or country == 'United Arab Emirates'

    msci_jump_results.append({
        'Country': country, 'T': T,
        'LR': LR, 'p_value': pval,
        'AIC_GARCH': -2*garch['loglik'] + 2*4,
        'AIC_Jump':  -2*garch_jmp['loglik'] + 2*7,
        'lambda': garch_jmp['params'][4],
        'theta':  garch_jmp['params'][5],
        'delta':  garch_jmp['params'][6],
        'alpha_drop': garch['params'][2] - garch_jmp['params'][2],
        'Oil_Exporter': is_oil,
    })
    sig = '***' if pval<0.01 else '**' if pval<0.05 else '*' if pval<0.1 else ''
    print(f'{country}: LR={LR:.1f} p={pval:.4f}{sig} λ={garch_jmp["params"][4]:.3f}')

msci_jump_df = pd.DataFrame(msci_jump_results)
print('\n', msci_jump_df.to_string())

### 3.5 Group comparisons

In [ ]:
for label, df in [('CDS', cds_jump_df), ('MSCI', msci_jump_df)]:
    print(f'\n{"="*60}')
    print(f'{label} — Mean jump parameters by group:')
    print(df.groupby('Oil_Exporter')[['LR', 'lambda', 'theta', 'delta', 'alpha_drop']].mean())

    exp  = df[df['Oil_Exporter']]
    ctrl = df[~df['Oil_Exporter']]

    for metric in ['LR', 'lambda', 'alpha_drop']:
        t, p = ttest_ind(exp[metric].dropna(), ctrl[metric].dropna())
        u, p_mw = mannwhitneyu(exp[metric].dropna(), ctrl[metric].dropna(), alternative='greater')
        print(f'  {metric}: Exp={exp[metric].mean():.4f} Ctrl={ctrl[metric].mean():.4f} '
              f't={t:.2f} p={p:.4f}  MW-U={u:.0f} p={p_mw:.4f}')

### 3.6 Co-jump analysis (Oil × CDS)

In [ ]:
@njit
def get_garch_variance(params, returns):
    mu, omega, alpha, beta = params
    T = len(returns)
    h = np.zeros(T)
    h[0] = np.var(returns)
    for t in range(1, T):
        h[t] = omega + alpha * (returns[t-1] - mu)**2 + beta * h[t-1]
    return h


def identify_jumps(returns, threshold=2.5):
    ret = returns.dropna().values
    garch = estimate_garch(ret)
    h = get_garch_variance(garch['params'], ret)
    mu = garch['params'][0]
    z = (ret - mu) / np.sqrt(h)
    return pd.Series(np.abs(z) > threshold, index=returns.dropna().index)


def cojump_test(oil_jumps, cds_jumps, country):
    common_idx = oil_jumps.index.intersection(cds_jumps.index)
    oil_j = oil_jumps.loc[common_idx].values
    cds_j = cds_jumps.loc[common_idx].values
    n = len(common_idx)
    a = ((oil_j) & (cds_j)).sum()
    b = ((~oil_j) & (cds_j)).sum()
    c = ((oil_j) & (~cds_j)).sum()
    d = ((~oil_j) & (~cds_j)).sum()
    n_oil = oil_j.sum()
    odds_ratio, fisher_p = fisher_exact([[a, b], [c, d]])
    cojump_rate = a / n_oil if n_oil > 0 else np.nan
    return {'Country': country, 'N': n, 'N_oil_jumps': int(n_oil),
            'N_cds_jumps': int(cds_j.sum()), 'N_cojumps': int(a),
            'Cojump_rate': cojump_rate, 'Odds_ratio': odds_ratio, 'Fisher_p': fisher_p}

In [ ]:
brent_returns = oil_returns['Brent']
oil_jumps = identify_jumps(brent_returns, threshold=2.5)
print(f'Oil (Brent) jumps: {oil_jumps.sum()} / {len(oil_jumps)} ({oil_jumps.mean()*100:.1f}%)')

cojump_results = []
for country in jump_countries:
    if country not in cds_returns.columns:
        continue
    ret = cds_returns[country].dropna()
    if len(ret) < 200 or (ret == 0).sum()/len(ret) > 0.3:
        continue
    cds_j = identify_jumps(ret, threshold=2.5)
    cojump_results.append(cojump_test(oil_jumps, cds_j, country))

cojump_df = pd.DataFrame(cojump_results)
cojump_df['Oil_Exporter'] = cojump_df['Country'].isin(jump_oil_exporters)
print('\n', cojump_df.to_string(index=False))

# Group comparison
print('\nMean co-jump rate by group:')
print(cojump_df.groupby('Oil_Exporter')[['Cojump_rate', 'Odds_ratio']].mean())

exp_cj  = cojump_df[cojump_df['Oil_Exporter']]['Cojump_rate'].dropna()
ctrl_cj = cojump_df[~cojump_df['Oil_Exporter']]['Cojump_rate'].dropna()
t, p = ttest_ind(exp_cj, ctrl_cj)
u, p_mw = mannwhitneyu(exp_cj, ctrl_cj, alternative='greater')
print(f'\nT-test: t={t:.3f} p={p:.4f}')
print(f'MW-U:   U={u:.0f} p={p_mw:.4f}')

## 4.2 Brent jump estimation and OVX → λ calibration

Daily jump detection on Brent (|z-score| > 2.5), weekly aggregation, jump-size distribution,
sigmoid calibration of the jump intensity λ as a function of OVX, and diagnostic plots.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.stats import norm
from scipy.optimize import curve_fit


# =========================================================
# PARAMETERS
# =========================================================
OIL_COL    = 'Brent'
OVX_COL    = 'OVXCLS'
Z_THRESH   = -2.5     # standardized threshold for jump detection
VOL_WINDOW = 252      # rolling window for daily σ̂
ANN_FACTOR = 52       # annualization for weekly λ

In [ ]:
# =========================================================
# 0)  LOAD & PREP
# =========================================================
oil_price = pd.read_csv('../data/processed/Oil/oil_prices_datastream.csv')
ovx       = pd.read_csv('../data/processed/Macroeconomic_variables/OVXCLS.csv')

oil_price['date'] = pd.to_datetime(oil_price['date'], errors='coerce')
ovx['date']       = pd.to_datetime(ovx['date'],       errors='coerce')

df = (pd.merge(oil_price[['date', OIL_COL]], ovx[['date', OVX_COL]],
               on='date', how='inner')
        .sort_values('date')
        .reset_index(drop=True))
df.ffill(inplace=True)

df = df[(df['date']>'2013-01-01') & (df['date']<'2024-12-31')]

In [ ]:
# =========================================================
# 1)  DAILY RETURNS
# =========================================================
df['log_ret']    = np.log(df[OIL_COL] / df[OIL_COL].shift(1))
df['arith_ret']  = df[OIL_COL] / df[OIL_COL].shift(1) - 1
df['roll_sigma'] = df['log_ret'].rolling(VOL_WINDOW, min_periods=126).std()
df['z_score']    = df['log_ret'] / df['roll_sigma']
df.dropna(subset=['log_ret', 'roll_sigma', OVX_COL], inplace=True)

In [ ]:
# =========================================================
# 2)  JUMP DETECTION  —  daily |z-score| > 2.5
# =========================================================
df['jump_daily'] = (np.abs(df['z_score']) > np.abs(Z_THRESH)).astype(int)
df['week']       = df['date'].dt.to_period('W')
jump_weeks_set   = set(df.loc[df['jump_daily'] == 1, 'week'])

print(f"Daily jump observations : {df['jump_daily'].sum()}")
print(f"  Negative jumps: {(df['z_score'] < Z_THRESH).sum()}")
print(f"  Positive jumps: {(df['z_score'] > -Z_THRESH).sum()}")
print(f"Weeks containing a jump : {len(jump_weeks_set)}")

In [ ]:
# =========================================================
# 3)  WEEKLY AGGREGATION
# =========================================================
# oil: last price of the week
oil_weekly = (df.set_index('date')[[OIL_COL]]
                .resample('W-FRI').last()
                .reset_index())
oil_weekly['week_ret'] = oil_weekly[OIL_COL] / oil_weekly[OIL_COL].shift(1) - 1

# OVX: last value of the week
ovx_weekly = (df.set_index('date')[[OVX_COL]]
                .resample('W-FRI').last()
                .reset_index())

df_weekly = pd.merge(oil_weekly, ovx_weekly, on='date', how='inner')
df_weekly['week']         = df_weekly['date'].dt.to_period('W')
df_weekly['is_jump_week'] = df_weekly['week'].isin(jump_weeks_set).astype(int)
df_weekly.dropna(subset=['week_ret', OVX_COL], inplace=True)


In [ ]:
# =========================================================
# 4)  JUMP SIZE DISTRIBUTION  —  all jump weeks
# =========================================================
from scipy.stats import shapiro, kstest, norm as sp_norm

jump_week_returns = df_weekly.loc[df_weekly['is_jump_week'] == 1, 'week_ret']

mu_J    = jump_week_returns.mean()
sigma_J = jump_week_returns.std()

print(f"Jump week returns — mean: {mu_J:.4f}  std: {sigma_J:.4f}")
print(f"N jump weeks: {len(jump_week_returns)}")
print(jump_week_returns.describe())

# test normality of weekly returns directly
ks_stat, ks_p   = kstest(jump_week_returns, 'norm', args=(mu_J, sigma_J))
sw_stat, sw_p   = shapiro(jump_week_returns)

print(f"\nNormality of weekly jump returns:")
print(f"  KS test:          stat={ks_stat:.4f},  p={ks_p:.4f}")
print(f"  Shapiro-Wilk:     stat={sw_stat:.4f},  p={sw_p:.4f}")

# also test ln(1+J) — what Merton (1976) actually requires
log_jump_rets = np.log(1 + jump_week_returns)
mu_logJ    = log_jump_rets.mean()
sigma_logJ = log_jump_rets.std()

ks_log_stat, ks_log_p = kstest(log_jump_rets, 'norm', args=(mu_logJ, sigma_logJ))
sw_log_stat, sw_log_p = shapiro(log_jump_rets)

print(f"\nNormality of ln(1+J) — Merton (1976) requirement:")
print(f"  KS test:          stat={ks_log_stat:.4f},  p={ks_log_p:.4f}")
print(f"  Shapiro-Wilk:     stat={sw_log_stat:.4f},  p={sw_log_p:.4f}")

# plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

x = np.linspace(jump_week_returns.min(), jump_week_returns.max(), 200)
axes[0].hist(jump_week_returns, bins=20, density=True, alpha=0.5, color='steelblue')
axes[0].plot(x, sp_norm.pdf(x, mu_J, sigma_J), lw=2, color='red',
             label=f'N({mu_J:.3f}, {sigma_J:.3f}²)')
axes[0].set_title('Weekly jump returns')
axes[0].set_xlabel('Arithmetic return')
axes[0].legend(fontsize=8)

x2 = np.linspace(log_jump_rets.min(), log_jump_rets.max(), 200)
axes[1].hist(log_jump_rets, bins=20, density=True, alpha=0.5, color='steelblue')
axes[1].plot(x2, sp_norm.pdf(x2, mu_logJ, sigma_logJ), lw=2, color='red',
             label=f'N({mu_logJ:.3f}, {sigma_logJ:.3f}²)')
axes[1].set_title('ln(1+J) — Merton requirement')
axes[1].set_xlabel('Log return')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

# keep J as mean for pricer
J = mu_J
print(f"\nJ (mean, all jump weeks): {J:.4f}")
print(f"σ_J:                      {sigma_J:.4f}")

In [ ]:
def sigmoid(x, L, k, x0):
    return L / (1 + np.exp(-k * (x - x0)))

df_weekly['realized_lam'] = df_weekly['is_jump_week'].rolling(52, min_periods=26).mean()
df_weekly['OVX_avg']      = df_weekly[OVX_COL].rolling(52, min_periods=26).mean()

df_sig = df_weekly.dropna(subset=['realized_lam', 'OVX_avg']).copy()

popt, _ = curve_fit(
    sigmoid,
    df_sig['OVX_avg'].values,
    df_sig['realized_lam'].values,
    p0=[0.15, 0.1, 60],
    bounds=([0, 0, 40], [1, 1, 200]),  # force x0 >= 40
    maxfev=10000
)

L, k, x0 = popt
print(f"L={L:.4f}, k={k:.4f}, x0={x0:.1f}")
print(f"λ_max annualized: {L*52:.1f}")

for ovx in [15, 20, 30, 36, 45, 60, 80, 100]:
    print(f"OVX={ovx:>3}  λ_annual={sigmoid(ovx,L,k,x0)*52:>5.1f}")

def ovx_to_lambda(ovx):
    if np.isnan(ovx) or ovx <= 0:
        return 0.0
    return float(sigmoid(ovx, L, k, x0) * 52)

df_weekly['lam_annual'] = df_weekly[OVX_COL].apply(ovx_to_lambda)

In [ ]:
# =========================================================
# 8)  DIAGNOSTIC PLOTS
# =========================================================

# add lambda to df_weekly for panel 4
df_weekly['lam_annual'] = df_weekly[OVX_COL].apply(ovx_to_lambda)

fig, axes = plt.subplots(4, 1, figsize=(13, 14), sharex=False)

date_min = df['date'].min()
date_max = df['date'].max()

# separate positive and negative jump days
neg_jump_days = df[(df['jump_daily'] == 1) & (df['z_score'] < 0)]
pos_jump_days = df[(df['jump_daily'] == 1) & (df['z_score'] > 0)]

# separate positive and negative jump weeks for panel 3
neg_jump_week_dates = set()
pos_jump_week_dates = set()
for _, row in df[df['jump_daily'] == 1].iterrows():
    if row['z_score'] < 0:
        neg_jump_week_dates.add(row['week'])
    else:
        pos_jump_week_dates.add(row['week'])

# ── Panel 1: Oil price with jump days ────────────────────
ax = axes[0]
ax.plot(df['date'], df[OIL_COL], lw=0.8, color='steelblue', zorder=2)
ax.scatter(neg_jump_days['date'], neg_jump_days[OIL_COL],
           color='red',   s=12, zorder=5, label='Negative jump day')
ax.scatter(pos_jump_days['date'], pos_jump_days[OIL_COL],
           color='green', s=12, zorder=5, label='Positive jump day')
ax.set_ylabel('Brent (USD)', fontsize=9)
ax.set_title('Brent crude oil price — detected jump days', fontsize=10)
ax.legend(fontsize=8, loc='upper right')
ax.set_xlim(date_min, date_max)
ax.grid(axis='y', lw=0.4, alpha=0.4)

# ── Panel 2: Daily z-scores ───────────────────────────────
ax = axes[1]
ax.plot(df['date'], df['z_score'], lw=0.5, color='gray', alpha=0.6, zorder=2)
ax.axhline(-abs(Z_THRESH), color='red',   ls='--', lw=1, label=f'z = {Z_THRESH}')
ax.axhline( abs(Z_THRESH), color='green', ls='--', lw=1, label=f'z = +{abs(Z_THRESH)}')
ax.axhline(0, color='black', lw=0.4, alpha=0.4)
ax.scatter(neg_jump_days['date'], neg_jump_days['z_score'],
           color='red',   s=12, zorder=5)
ax.scatter(pos_jump_days['date'], pos_jump_days['z_score'],
           color='green', s=12, zorder=5)
ax.set_ylabel('z-score', fontsize=9)
ax.set_title('Standardized daily returns (rolling 252-day σ)', fontsize=10)
ax.legend(fontsize=8, loc='upper right')
ax.set_xlim(date_min, date_max)
ax.grid(axis='y', lw=0.4, alpha=0.4)

# ── Panel 3: Weekly OVX + jump week flags ─────────────────
ax = axes[2]
ax.plot(df_weekly['date'], df_weekly[OVX_COL],
        lw=0.8, color='darkorange', zorder=3, label='OVX')

for d in df_weekly.loc[df_weekly['week'].isin(neg_jump_week_dates), 'date']:
    ax.axvspan(d - pd.Timedelta(days=3), d + pd.Timedelta(days=3),
               color='red', alpha=0.15, lw=0)

for d in df_weekly.loc[df_weekly['week'].isin(pos_jump_week_dates), 'date']:
    ax.axvspan(d - pd.Timedelta(days=3), d + pd.Timedelta(days=3),
               color='green', alpha=0.15, lw=0)

from matplotlib.patches import Patch
ax.legend(handles=[
    plt.Line2D([0], [0], color='darkorange', lw=1.5, label='OVX'),
    Patch(facecolor='red',   alpha=0.3, label='Negative jump week'),
    Patch(facecolor='green', alpha=0.3, label='Positive jump week'),
], fontsize=8, loc='upper right')

ax.set_ylabel('OVX', fontsize=9)
ax.set_title('Weekly OVX and detected jump weeks', fontsize=10)
ax.set_xlim(df_weekly['date'].min(), df_weekly['date'].max())
ax.grid(axis='y', lw=0.4, alpha=0.4)

# ── Panel 4: Time-varying λ — sigmoid specification ───────
df_plot = df_weekly.dropna(subset=['lam_annual'])
ax = axes[3]
ax.plot(df_plot['date'], df_plot['lam_annual'],
        lw=0.8, color='purple', zorder=3)
ax.axhline(df_plot['lam_annual'].mean(), color='purple', ls='--', lw=0.8,
           alpha=0.6, label=f"Mean λ = {df_plot['lam_annual'].mean():.1f}")
ax.axhline(L * 52, color='black', ls=':', lw=0.8, alpha=0.5,
           label=f'Natural max = {L*52:.1f}')
ax.fill_between(df_plot['date'], df_plot['lam_annual'],
                df_plot['lam_annual'].mean(),
                where=df_plot['lam_annual'] > df_plot['lam_annual'].mean(),
                color='purple', alpha=0.15)
ax.set_ylabel('λ (annualized)', fontsize=9)
ax.set_title(
    f'Time-varying jump intensity — sigmoid fit '
    f'(L={L:.4f}, k={k:.4f}, x₀={x0:.1f})',
    fontsize=10)
ax.legend(fontsize=8, loc='upper right')
ax.set_xlim(df_weekly['date'].min(), df_weekly['date'].max())
ax.grid(axis='y', lw=0.4, alpha=0.4)

# ── shared x-formatting ───────────────────────────────────
import matplotlib.dates as mdates
for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator())
    plt.setp(ax.xaxis.get_majorticklabels(), fontsize=8)

plt.tight_layout(h_pad=1.5)
plt.savefig('../output/figures/jump_estimation_diagnostic.png', dpi=150, bbox_inches='tight')
plt.show()# =========================================================
# 8)  DIAGNOSTIC PLOTS
# =========================================================

# add lambda to df_weekly for panel 4
df_weekly['lam_annual'] = df_weekly[OVX_COL].apply(ovx_to_lambda)

fig, axes = plt.subplots(4, 1, figsize=(13, 14), sharex=False)

date_min = df['date'].min()
date_max = df['date'].max()

# separate positive and negative jump days
neg_jump_days = df[(df['jump_daily'] == 1) & (df['z_score'] < 0)]
pos_jump_days = df[(df['jump_daily'] == 1) & (df['z_score'] > 0)]

# separate positive and negative jump weeks for panel 3
neg_jump_week_dates = set()
pos_jump_week_dates = set()
for _, row in df[df['jump_daily'] == 1].iterrows():
    if row['z_score'] < 0:
        neg_jump_week_dates.add(row['week'])
    else:
        pos_jump_week_dates.add(row['week'])

# ── Panel 1: Oil price with jump days ────────────────────
ax = axes[0]
ax.plot(df['date'], df[OIL_COL], lw=0.8, color='steelblue', zorder=2)
ax.scatter(neg_jump_days['date'], neg_jump_days[OIL_COL],
           color='red',   s=12, zorder=5, label='Negative jump day')
ax.scatter(pos_jump_days['date'], pos_jump_days[OIL_COL],
           color='green', s=12, zorder=5, label='Positive jump day')
ax.set_ylabel('Brent (USD)', fontsize=9)
ax.set_title('Brent crude oil price — detected jump days', fontsize=10)
ax.legend(fontsize=8, loc='upper right')
ax.set_xlim(date_min, date_max)
ax.grid(axis='y', lw=0.4, alpha=0.4)

# ── Panel 2: Daily z-scores ───────────────────────────────
ax = axes[1]
ax.plot(df['date'], df['z_score'], lw=0.5, color='gray', alpha=0.6, zorder=2)
ax.axhline(-abs(Z_THRESH), color='red',   ls='--', lw=1, label=f'z = {Z_THRESH}')
ax.axhline( abs(Z_THRESH), color='green', ls='--', lw=1, label=f'z = +{abs(Z_THRESH)}')
ax.axhline(0, color='black', lw=0.4, alpha=0.4)
ax.scatter(neg_jump_days['date'], neg_jump_days['z_score'],
           color='red',   s=12, zorder=5)
ax.scatter(pos_jump_days['date'], pos_jump_days['z_score'],
           color='green', s=12, zorder=5)
ax.set_ylabel('z-score', fontsize=9)
ax.set_title('Standardized daily returns (rolling 252-day σ)', fontsize=10)
ax.legend(fontsize=8, loc='upper right')
ax.set_xlim(date_min, date_max)
ax.grid(axis='y', lw=0.4, alpha=0.4)

# ── Panel 3: Weekly OVX + jump week flags ─────────────────
ax = axes[2]
ax.plot(df_weekly['date'], df_weekly[OVX_COL],
        lw=0.8, color='darkorange', zorder=3, label='OVX')

for d in df_weekly.loc[df_weekly['week'].isin(neg_jump_week_dates), 'date']:
    ax.axvspan(d - pd.Timedelta(days=3), d + pd.Timedelta(days=3),
               color='red', alpha=0.15, lw=0)

for d in df_weekly.loc[df_weekly['week'].isin(pos_jump_week_dates), 'date']:
    ax.axvspan(d - pd.Timedelta(days=3), d + pd.Timedelta(days=3),
               color='green', alpha=0.15, lw=0)

from matplotlib.patches import Patch
ax.legend(handles=[
    plt.Line2D([0], [0], color='darkorange', lw=1.5, label='OVX'),
    Patch(facecolor='red',   alpha=0.3, label='Negative jump week'),
    Patch(facecolor='green', alpha=0.3, label='Positive jump week'),
], fontsize=8, loc='upper right')

ax.set_ylabel('OVX', fontsize=9)
ax.set_title('Weekly OVX and detected jump weeks', fontsize=10)
ax.set_xlim(df_weekly['date'].min(), df_weekly['date'].max())
ax.grid(axis='y', lw=0.4, alpha=0.4)

# ── Panel 4: Time-varying λ — sigmoid specification ───────
df_plot = df_weekly.dropna(subset=['lam_annual'])
ax = axes[3]
ax.plot(df_plot['date'], df_plot['lam_annual'],
        lw=0.8, color='purple', zorder=3)
ax.axhline(df_plot['lam_annual'].mean(), color='purple', ls='--', lw=0.8,
           alpha=0.6, label=f"Mean λ = {df_plot['lam_annual'].mean():.1f}")
ax.axhline(L * 52, color='black', ls=':', lw=0.8, alpha=0.5,
           label=f'Natural max = {L*52:.1f}')
ax.fill_between(df_plot['date'], df_plot['lam_annual'],
                df_plot['lam_annual'].mean(),
                where=df_plot['lam_annual'] > df_plot['lam_annual'].mean(),
                color='purple', alpha=0.15)
ax.set_ylabel('λ (annualized)', fontsize=9)
ax.set_title(
    f'Time-varying jump intensity — sigmoid fit '
    f'(L={L:.4f}, k={k:.4f}, x₀={x0:.1f})',
    fontsize=10)
ax.legend(fontsize=8, loc='upper right')
ax.set_xlim(df_weekly['date'].min(), df_weekly['date'].max())
ax.grid(axis='y', lw=0.4, alpha=0.4)

# ── shared x-formatting ───────────────────────────────────
import matplotlib.dates as mdates
for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator())
    plt.setp(ax.xaxis.get_majorticklabels(), fontsize=8)

plt.tight_layout(h_pad=1.5)
plt.show()